# Problem Statement
News content is generated in massive amounts daily, making manual categorization into topics like World, Sports, Business, and Sci/Tech time-consuming and inefficient. This project addresses the need for an automated system that can accurately classify news headlines using a BERT-based deep learning model.
# Objectives
## Dataset Loading and Preprocessing
Load the AG News dataset using Hugging Face.
Clean and tokenize text data using a BERT tokenizer.
Convert text into numerical format suitable for model training.
## Model Development and Training
Use a pre-trained BERT (bert-base-uncased).
Fine-tune the model on the AG News dataset for text classification.
Train the model using the PyTorch and Hugging Face Trainer.
## Evaluation with Relevant Metrics
Evaluate the model using:
Accuracy
F1-score
Analyze performance using the Evaluate and scikit-learn metrics.
## Deployment
Deploy the trained model using Streamlit or Gradio.
Allow users to input a headline and get real-time predictions.
## Final Summary
This project successfully builds an end-to-end news classification system using BERT, covering dataset preprocessing, model fine-tuning, evaluation, and deployment. It demonstrates how modern NLP techniques can automate text classification tasks with high accuracy, making news categorization faster and more efficient.

Here we Installs all required Python libraries to build, train, evaluate, and deploy your BERT-based news classification project.

In [ ]:
pip install transformers datasets evaluate torch scikit-learn gradio streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 68.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 83.3 MB/s eta 0:00:00


 Here I Imports all required libraries and tools for loading data, preprocessing text, building, training, and evaluating a BERT-based news classification model.

In [ ]:
import os
import numpy as np
import evaluate
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

Here I Loads the AG News dataset from the library and prints a message to indicate the data loading process has started.

In [ ]:
# 1. Load Dataset
dataset = load_dataset("ag_news")

Here I Loads the BERT tokenizer, converts text data into tokenized format with a fixed length, applies it to the dataset, and prepares dynamic padding for efficient training.

In [ ]:
# 2. Tokenization
model_checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=128)
tokenized_datasets = dataset.map(tokenize_function, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

Here I Defines label mappings and loads a pre-trained BERT model configured for classifying text into four news categories.

In [ ]:
# 3. Model Setup with Label Mapping
id2label = {0: "World", 1: "Sports", 2: "Business", 3: "Sci/Tech"}
label2id = {"World": 0, "Sports": 1, "Business": 2, "Sci/Tech": 3}
model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint,
    num_labels=4,
    id2label=id2label,
    label2id=label2id
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Here I Loads accuracy and F1-score metrics, computes predictions from model outputs, and returns both evaluation scores to measure model performance.

In [ ]:
# 4. Metrics
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    accuracy = accuracy_metric.compute(predictions=predictions, references=labels)
    f1 = f1_metric.compute(predictions=predictions, references=labels, average="weighted")
    return {
        "accuracy": accuracy["accuracy"],
        "f1": f1["f1"]
    }

Here I Defines the training configuration, including learning rate, batch size, epochs, evaluation strategy, and model saving settings.

In [ ]:
# 5. Training Arguments
training_args = TrainingArguments(
    output_dir="./bert_news_classifier",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none",
)

Here I Initializes the Trainer by connecting the model, training settings, datasets, tokenizer, and evaluation function to handle training and evaluation automatically.

In [ ]:
# 6. Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"].shuffle(seed=42).select(range(5000)),
    eval_dataset=tokenized_datasets["test"].select(range(1000)),
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

Here I Starts the training process and fine-tunes the model on the dataset.

In [ ]:
# 7. Execute Training
print("Starting training on v5.3.0...")
trainer.train()

Starting training on v5.3.0...


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.338576,0.893000,0.891874
2,0.366900,0.328581,0.896000,0.895467


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=626, training_loss=0.33403618114824873, metrics={'train_runtime': 219.3997, 'train_samples_per_second': 45.579, 'train_steps_per_second': 2.853, 'total_flos': 473164396293888.0, 'train_loss': 0.33403618114824873, 'epoch': 2.0})

Here I Defines a function that takes input text, processes it through the trained model, and returns the predicted news category.

In [ ]:
def predict(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    # Move inputs to the same device as the model
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    outputs = model(**inputs)
    logits = outputs.logits
    predicted_class_id = logits.argmax().item()
    return model.config.id2label[predicted_class_id]
# Example to see result:
print(predict("Apple releases new iPhone with AI features"))

Sci/Tech


I Creates a simple web app using Streamlit where users can enter a news headline and get the predicted category displayed but this not work in the Google Colab and Jupyter Notebook

In [ ]:
import streamlit as st
import torch
st.title("News Topic Classifier (BERT)")
user_input = st.text_input("Enter news headline:")
if st.button("Predict"):
    if user_input:
        prediction = predict(user_input)
        st.success(f"Predicted Category: {prediction}")
    else:
        st.warning("Please enter a news headline.")

2026-03-24 12:03:09.054 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-24 12:03:09.060 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-24 12:03:09.062 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-24 12:03:09.065 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-24 12:03:09.067 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-24 12:03:09.072 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-24 12:03:09.077 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-24 12:03:09.078 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar